In [12]:
import torch
import torch.nn as nn

class ResNetSim(nn.Module):
    def __init__(self):
        super(ResNetSim, self).__init__()
        # 1) stem
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)  # -> (B,64,320,320)
        self.pool  = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)                  # -> (B,64,160,160)

        # 2) conv2_x: six 3×3, 64 -> 64 (same H×W)
        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.conv3 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.conv4 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.conv5 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.conv6 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.conv7 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)

        # 3) conv3_x: first block downsamples to 80×80, then keep size
        self.conv8  = nn.Conv2d(64, 128, 3, 2, 1, bias=False)   # -> (B,128,80,80)
        self.conv9  = nn.Conv2d(128, 128, 3, 1, 1, bias=False)
        self.conv10 = nn.Conv2d(128, 128, 3, 1, 1, bias=False)
        self.conv11 = nn.Conv2d(128, 128, 3, 1, 1, bias=False)
        self.conv12 = nn.Conv2d(128, 128, 3, 1, 1, bias=False)
        self.conv13 = nn.Conv2d(128, 128, 3, 1, 1, bias=False)
        self.conv14 = nn.Conv2d(128, 128, 3, 1, 1, bias=False)
        self.conv15 = nn.Conv2d(128, 128, 3, 1, 1, bias=False)

        # 4) conv4_x: downsample to 40×40 then keep size
        self.conv16 = nn.Conv2d(128, 256, 3, 2, 1, bias=False)  # -> (B,256,40,40)
        self.conv17 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv18 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv19 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv20 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv21 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv22 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv23 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv24 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv25 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv26 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)
        self.conv27 = nn.Conv2d(256, 256, 3, 1, 1, bias=False)

        # 5) conv5_x (28→33): downsample to 20×20, then keep size
        self.conv28 = nn.Conv2d(256, 512, 3, 2, 1, bias=False)  # -> (B,512,20,20)
        self.conv29 = nn.Conv2d(512, 512, 3, 1, 1, bias=False)
        self.conv30 = nn.Conv2d(512, 512, 3, 1, 1, bias=False)
        self.conv31 = nn.Conv2d(512, 512, 3, 1, 1, bias=False)
        self.conv32 = nn.Conv2d(512, 512, 3, 1, 1, bias=False)
        self.conv33 = nn.Conv2d(512, 512, 3, 1, 1, bias=False)

        # remove for detr model
        self.avg = nn.AdaptiveAvgPool2d((1,1))
        self.fc  = nn.Linear(512, 1000)

    def forward(self, x):
        x = self.conv1(x)    # (B,64,320,320)
        x = self.pool(x)     # (B,64,160,160)

        x = self.conv2(x);  x = self.conv3(x);  x = self.conv4(x)
        x = self.conv5(x);  x = self.conv6(x);  x = self.conv7(x)

        x = self.conv8(x)    # (B,128,80,80)
        x = self.conv9(x);   x = self.conv10(x); x = self.conv11(x); x = self.conv12(x)
        x = self.conv13(x);  x = self.conv14(x); x = self.conv15(x)

        x = self.conv16(x)   # (B,256,40,40)
        x = self.conv17(x);  x = self.conv18(x); x = self.conv19(x); x = self.conv20(x)
        x = self.conv21(x);  x = self.conv22(x); x = self.conv23(x); x = self.conv24(x)
        x = self.conv25(x);  x = self.conv26(x); x = self.conv27(x)

        x = self.conv28(x)   # (B,512,20,20)
        x = self.conv29(x);  x = self.conv30(x); x = self.conv31(x)
        x = self.conv32(x);  x = self.conv33(x)

        # remove for detr model
        x = self.avg(x)            # (B,512,20,20) -> (B,512,1,1)
        x = torch.flatten(x, 1)    # (B,512)
        x = self.fc(x)             # (B,1000)
        return x

# demo
model = ResNetSim()
inp = torch.randn(1, 3, 640, 640)
out = model(inp)
print(out.shape)  # -> torch.Size([1, 512, 20, 20])


torch.Size([1, 1000])
